# P1: Framing a Data Problem
**DSA 405 · Fall 2026 · Nicole Brockmole · 200684137**

## 1. The question

**Do S&P 500 companies whose 10-K risk factor section (Item 1A) got a lot longer from one year to the next have worse stock returns over the following 12 months than companies whose risk section stayed about the same length?**

Why I care about this: every public company has to list its risks in the 10-K, but nobody tells them how much to write. So when a company suddenly adds a bunch of new risk language, I'm curious whether that actually means something. It could be management seeing problems coming, or it could just be the legal team covering themselves. I'm interested in a career in financial analysis, and analysts read these filings every year, so if the length change predicts returns that's a free signal sitting in public data. If it doesn't predict anything, that's still worth knowing because it means people shouldn't read too much into it.

The table that would answer this: one row per company per year, with (a) the % change in Item 1A word count compared to the year before and (b) the stock's return over the 12 months after the filing date. Then split companies into five groups based on (a) and compare the average return in each group.

## 2. Sources & access evidence

### Source 1: SEC EDGAR 10-K filings (web source, API)
| | |
|---|---|
| **Publisher** | U.S. Securities and Exchange Commission (SEC) |
| **URL** | Submissions API: `https://data.sec.gov/submissions/CIK{10-digit CIK}.json` ; filing documents under `https://www.sec.gov/Archives/edgar/data/{CIK}/{accession}/` |
| **Coverage & time span** | Every 10-K filed by every registrant; I'll pull S&P 500 companies, fiscal years 2019–2024 |
| **Approximate size** | ~500 companies × 6 years ≈ 3,000 filings, roughly 100–300 pages each |
| **Access method** | API (JSON) for the filing index, then direct download of the 10-K documents |

### Source 2: Daily stock prices (Stooq)
| | |
|---|---|
| **Publisher** | Stooq (stooq.com), a free financial data site |
| **URL** | `https://stooq.com/q/d/l/?s={ticker}.us&i=d` (one CSV per ticker, e.g. `https://stooq.com/q/d/l/?s=mmm.us&i=d`) |
| **Coverage & time span** | Daily open/high/low/close/volume for U.S. stocks, 20+ years; I need 2019–2025 |
| **Approximate size** | ~500 tickers × ~1,750 trading days ≈ 875,000 rows |
| **Access method** | download (`read_csv` straight from the URL) |

### Source 3: S&P 500 constituent list (Wikipedia)
| | |
|---|---|
| **Publisher** | Wikipedia (Wikimedia Foundation) |
| **URL** | `https://en.wikipedia.org/wiki/List_of_S%26P_500_companies` |
| **Coverage & time span** | Current S&P 500 members with ticker, company name, sector, and CIK |
| **Approximate size** | ~503 rows |
| **Access method** | `read_html` |

How they connect: Wikipedia gives me both the ticker and the CIK for each company. EDGAR uses CIK, Stooq uses ticker, so Wikipedia is the bridge between the two.

In [1]:

import pandas as pd
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "DSA405StudentProject/1.0 (academic research; contact: nmbrockm@ncsu.edu)"
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

print("Status code:", response.status_code)
print("Response size:", len(response.content))

response.raise_for_status()

tables = pd.read_html(StringIO(response.text))

print("Number of tables:", len(tables))

sp500 = next(
    table for table in tables
    if {"Symbol", "CIK"}.issubset(table.columns)
)

print("S&P 500 rows:", len(sp500))
display(sp500.head())


Status code: 200
Response size: 572904
Number of tables: 2
S&P 500 rows: 503


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [2]:

HEADERS = {
    "User-Agent": "DSA405StudentProject/1.0 (academic research; contact: YOUR_EMAIL)"
}


In [3]:
cik = str(sp500.loc[0, "CIK"]).zfill(10)
r = requests.get(f"https://data.sec.gov/submissions/CIK{cik}.json", headers=HEADERS)
print("EDGAR status code:", r.status_code)

filings = pd.DataFrame(r.json()["filings"]["recent"])
tenks = filings[filings["form"] == "10-K"]
print(sp500.loc[0, "Security"], "— 10-K filings in recent index:", len(tenks))
tenks[["form", "filingDate", "accessionNumber"]].head()

EDGAR status code: 200
3M — 10-K filings in recent index: 8


,form,filingDate,accessionNumber
119,10-K,2026-02-03,0000066740-26-000014
222,10-K,2025-02-05,0000066740-25-000006
335,10-K,2024-02-07,0000066740-24-000016
456,10-K,2023-02-08,0000066740-23-000014
599,10-K,2022-02-09,0000066740-22-000010


In [4]:

from io import StringIO

ticker = "MMM"
stooq_url = "https://stooq.com/q/d/l/?s=mmm.us&i=d"

try:
    r2 = requests.get(
        stooq_url,
        headers=HEADERS,
        timeout=30
    )

    print("Stooq status code:", r2.status_code)
    print("Response preview:", r2.text[:200])

    if r2.status_code == 200 and r2.text.lstrip().startswith("Date,"):
        prices = pd.read_csv(StringIO(r2.text))
        print("Daily price rows:", len(prices))
        print("Date range:", prices["Date"].min(), "to", prices["Date"].max())
        display(prices.head())
    else:
        print("No valid price CSV returned.")

except requests.RequestException as error:
    print("Stooq request failed:", error)


Stooq status code: 200
Response preview: <!DOCTYPE html><html><head><meta charset="utf-8"><meta name="robots" content="noindex,nofollow"></head><body><noscript>This site requires JavaScript to verify your browser. Please enable JavaScript an
No valid price CSV returned.


**Data access issue:** Stooq ended up being the biggest problem with my original plan. When I tested the CSV download, the website returned a browser-verification page instead of stock prices. I also found that its robots.txt restricts automated crawling. Because of that, I decided not to keep trying to access it through Python. I would need to find another approved stock-price source to complete the original analysis.

## 3. Constraints & guardrails

The cell below pulls the robots.txt for Stooq and Wikipedia so the actual lines show up in the notebook. The SEC one wouldn't load from Python (403), so I opened it in a browser and quoted it by hand.

In [5]:
for site in ["https://stooq.com/robots.txt", "https://en.wikipedia.org/robots.txt"]:
    r = requests.get(site, headers=HEADERS)
    print("=" * 70)
    print(site, "— status", r.status_code)
    print(r.text[:1500])

https://stooq.com/robots.txt — status 200
User-agent: Bingbot
Allow: /

User-agent: Googlebot
Allow: /

User-agent: *
Disallow: /

https://en.wikipedia.org/robots.txt — status 200
﻿# robots.txt for http://www.wikipedia.org/ and friends
#
# Please note: There are a lot of pages on this site, and there are
# some misbehaved spiders out there that go _way_ too fast. If you're
# irresponsible, your access to the site may be blocked.
#

# Observed spamming large amounts of https://en.wikipedia.org/?curid=NNNNNN
# and ignoring 429 ratelimit responses, claims to respect robots:
# http://mj12bot.com/
User-agent: MJ12bot
Disallow: /

# advertising-related bots:
User-agent: Mediapartners-Google*
Disallow: /

# Wikipedia work bots:
User-agent: IsraBot
Disallow:

User-agent: Orthogaffe
Disallow:

# Crawlers that are kind enough to obey, but which we'd rather not have
# unless they're feeding search engines.
User-agent: UbiCrawler
Disallow: /

User-agent: DOC
Disallow: /

User-agent: Zao
Disallow: 

### Source 1: SEC EDGAR
I tried accessing the SEC's robots.txt file through Python, but the request returned a 403 error. Instead, I used the SEC's published fair access guidelines to check how I should collect the data. The SEC provides a documented EDGAR API and asks users to identify themselves, limit their requests, and avoid overwhelming the website. I was able to access the filing information successfully, so I used the official API and followed those guidelines.

The SEC's "Accessing EDGAR Data" page (`https://www.sec.gov/search-filings/edgar-search-assistance/accessing-edgar-data`) has a "Fair access" section that says: *"Current max request rate: 10 requests/second."* It also tells you to "use efficient scripting," only download what you need, and "Please declare your user agent in request headers," with the example format `Sample Company Name AdminContact@<sample company domain>.com`. It says the SEC "does not allow botnets or automated tools to crawl the site," which I take to mean use the API and the documented file paths, don't crawl around the website. EDGAR filings are government public records so there's no license or attribution requirement other than saying the data came from the SEC. My R script already uses a User-Agent with my name and NCSU email and pauses between requests, and this notebook uses the same header.

### Source 2: Stooq
I tried accessing Stooq's daily stock-price data, but even though the request returned a 200 status code, it gave me a browser-verification page instead of an actual CSV file. I also checked Stooq's robots.txt, which says `User-agent: *` and `Disallow: /`. This means automated crawling is restricted, so I decided not to keep trying to download the data through Python. I'll need to use a different approved source for stock prices if I want to complete that part of my analysis.

### Source 3: Wikipedia
Wikipedia's robots.txt is printed in the cell above. The top of the file basically says there are a ton of pages and some bots go way too fast and might get blocked. The Disallow lines are all for special/admin pages (`/w/`, `/wiki/Special:`, etc.), not normal articles, so one `read_html` call on one article page is fine. The content is licensed **CC BY-SA 4.0** (it says at the bottom of every article: "Text is available under the Creative Commons Attribution-ShareAlike 4.0 License"), so I need to give credit, which I'll do by citing the page URL and the date I pulled it.

### Course guardrails
| Guardrail | How my project meets it |
|---|---|
| **Scrape only sandboxes, documented APIs / open-data licenses, or sites whose robots.txt + ToS permit it** | EDGAR is a documented API with published access guidelines, and Wikipedia provides openly licensed information. I checked Stooq's robots.txt and found that it restricts automated crawling, so I stopped trying to collect stock prices through Python. |
| **Never data behind a login or paywall** | All three are open. No account, no key, no payment. |
| **Never personal or identifiable information about individuals** | Everything is about public companies, not people. 10-Ks do name executives, but I'm only measuring how long the Item 1A section is, not saving anything about individuals. |
| **Always rate-limit, identify the scraper honestly, and cache** | For EDGAR, I used a User-Agent and limited how often my code made requests. I also saved downloaded data locally so I wouldn't have to keep requesting it. For Wikipedia, I made a single request. I stopped the automated Stooq download after discovering its restrictions. |

After checking Stooq's access restrictions, I realized I couldn't rely on it for automated stock-price downloads. For the next part of my project, I would need to find a permitted alternative or adjust my analysis based on the data I could collect.

## 4. Tier declaration

**Tier 3 — Stretch.** This needs a multi-page pull from EDGAR with rate limiting, which is Tier 2 on its own, plus pulling Item 1A out of raw 10-K text. That part is the hard part, because companies don't format their section headers the same way, so I'll need to find where 1A starts and ends and check that what I grabbed is actually the risk section (not empty, starts at 1A, stops at 1B or Item 2). I picked Tier 3 because I've already gotten part of the EDGAR download and text extraction working in R for a research assistant project, so it feels like a stretch I can actually pull off instead of a total reach.

If the section parsing ends up being easier than I expect, what would make me push it further is adding a many-to-many join, like matching companies to multiple SIC industry codes that change over time.

## Self-scored rubric

| # | Criterion | Wt | My score | Why |
|---|---|---|---|---|
| 1 | Question & motivation | ×1 | 4 | One sentence question, described the table that answers it, explained who it matters to |
| 2 | Sources & access evidence | ×2 | 4 | I included access checks for all three sources. Wikipedia and EDGAR worked, and I saved the status codes and results. Stooq returned a browser-verification page instead of stock prices, so I documented the problem and explained why I couldn't use it. |
| 3 | Constraints & guardrails | ×2 | 3–4 | SEC fair access policy quoted, robots.txt printed for Stooq and Wikipedia, Wikipedia license quoted, all four guardrails covered; Stooq's terms are uncertain and flagged |
| 4 | Tier declaration & fit | ×1 | 4 | Tier 3 explained against what I already know how to do, said what would make me raise it |